# BERTScore permutation control + re-scoring

Runs in Google Colab on a free GPU. Takes about 10-20 minutes.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

Then upload `bertscore_input.csv` when cell 2 asks for it.

At the end, download **`bertscore_results.csv`** and **`bertscore_summary.txt`** and send both back.
Do not edit `bertscore_input.csv` - the pairings in it are deliberate.

In [1]:
# Cell 1 - install and confirm the GPU is on
!pip -q install bert-score==0.3.13
import torch
print('torch', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. It will still run but may take over an hour.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
torch 2.11.0+cu128
GPU available: True
device: Tesla T4


In [2]:
# Cell 2 - upload bertscore_input.csv
from google.colab import files
import pandas as pd

uploaded = files.upload()          # pick bertscore_input.csv
df = pd.read_csv('bertscore_input.csv')
df['response']  = df['response'].fillna('').astype(str)
df['reference'] = df['reference'].fillna('').astype(str)
print(len(df), 'pairs loaded')
print(df['kind'].value_counts())
df.head(3)

Saving bertscore_input.csv to bertscore_input.csv
3600 pairs loaded
kind
correct            1200
permuted_global    1200
permuted_domain    1200
Name: count, dtype: int64


,pair_id,model,prompt,resp_row,ref_row,kind,domain,note,response,reference
0,1,Gemini,Zero-shot,2,2,correct,Mathematics,NaN,3.14,The value of π(pi) rounded to two decimal plac...
1,2,Gemini,Zero-shot,2,362,permuted_global,Mathematics,NaN,3.14,The Odd Couple
2,3,Gemini,Zero-shot,2,21,permuted_domain,Mathematics,NaN,3.14,The Lowest Common Multiple (LCM) of 12 and 18 ...


In [3]:
# Cell 3 - score everything. Two passes: raw, then baseline-rescaled.
import bert_score, torch

cands = df['response'].tolist()
refs  = df['reference'].tolist()

P, R, F = bert_score.score(cands, refs, lang='en', model_type='roberta-large',
                           rescale_with_baseline=False, batch_size=64, verbose=True)
df['P_raw'], df['R_raw'], df['F1_raw'] = P.numpy(), R.numpy(), F.numpy()
print('raw pass done')

Pr, Rr, Fr = bert_score.score(cands, refs, lang='en', model_type='roberta-large',
                              rescale_with_baseline=True, batch_size=64, verbose=True)
df['P_rescaled'], df['R_rescaled'], df['F1_rescaled'] = Pr.numpy(), Rr.numpy(), Fr.numpy()
print('rescaled pass done')

print('bert-score version:', bert_score.__version__)
import transformers; print('transformers version:', transformers.__version__)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/25 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/57 [00:00<?, ?it/s]

done in 14.04 seconds, 256.35 sentences/sec
rescaled pass done
bert-score version: 0.3.12
transformers version: 5.13.1


In [4]:
# Cell 4 - save a slim results file (no response text, so it is small enough to email)
keep = ['pair_id','model','prompt','resp_row','ref_row','kind','domain','note',
        'P_raw','R_raw','F1_raw','P_rescaled','R_rescaled','F1_rescaled']
df[keep].to_csv('bertscore_results.csv', index=False)
print('wrote bertscore_results.csv', len(df), 'rows')

wrote bertscore_results.csv 3600 rows


In [5]:
# Cell 5 - summary
import io, bert_score, transformers
buf = io.StringIO()
def w(s=''):
    print(s); buf.write(str(s) + '\n')

w('bert-score  : ' + bert_score.__version__)
w('transformers: ' + transformers.__version__)
w('model       : roberta-large')
w('pairs       : %d' % len(df))
w()

for col in ['F1_raw','F1_rescaled']:
    w('=' * 66)
    w(col)
    w('=' * 66)
    w('%-10s %-12s %8s %8s %8s %8s' % ('model','prompt','correct','perm_glb','perm_dom','gap'))
    for (m, p), g in df.groupby(['model','prompt'], sort=False):
        c  = g[g.kind == 'correct'][col].mean()
        pg = g[g.kind == 'permuted_global'][col].mean()
        pd_ = g[g.kind == 'permuted_domain'][col].mean()
        w('%-10s %-12s %8.4f %8.4f %8.4f %8.4f' % (m, p, c, pg, pd_, c - pd_))
    w()
    for k in ['correct','permuted_global','permuted_domain']:
        s = df[df.kind == k][col]
        w('%-16s pooled mean=%.4f  sd=%.4f  min=%.4f  p05=%.4f  p95=%.4f  max=%.4f'
          % (k, s.mean(), s.std(), s.min(), s.quantile(.05), s.quantile(.95), s.max()))
    # how often does a permuted pair outscore the median correct pair?
    med = df[df.kind == 'correct'][col].median()
    for k in ['permuted_global','permuted_domain']:
        s = df[df.kind == k][col]
        w('%-16s %.1f%% of permuted pairs score above the median correct pair (%.4f)'
          % (k, 100.0 * (s > med).mean(), med))
    w()

w('=' * 66)
w('Precision / recall on correctly paired responses (closes D3)')
w('=' * 66)
w('%-10s %-12s %8s %8s %8s' % ('model','prompt','P','R','F1'))
cor = df[df.kind == 'correct']
for (m, p), g in cor.groupby(['model','prompt'], sort=False):
    w('%-10s %-12s %8.4f %8.4f %8.4f' % (m, p, g.P_raw.mean(), g.R_raw.mean(), g.F1_raw.mean()))

open('bertscore_summary.txt','w').write(buf.getvalue())
print('\nwrote bertscore_summary.txt')

bert-score  : 0.3.12
transformers: 5.13.1
model       : roberta-large
pairs       : 3600

F1_raw
model      prompt        correct perm_glb perm_dom      gap
Gemini     Zero-shot      0.8978   0.8289   0.8397   0.0581
Gemini     CoT            0.8552   0.8164   0.8228   0.0324
Gemini     Structured     0.8615   0.8159   0.8242   0.0373

correct          pooled mean=0.8715  sd=0.0349  min=0.7757  p05=0.8254  p95=0.9366  max=1.0000
permuted_global  pooled mean=0.8204  sd=0.0188  min=0.7493  p05=0.7903  p95=0.8501  max=0.9100
permuted_domain  pooled mean=0.8289  sd=0.0192  min=0.7555  p05=0.8001  p95=0.8598  max=0.9700
permuted_global  1.2% of permuted pairs score above the median correct pair (0.8658)
permuted_domain  3.5% of permuted pairs score above the median correct pair (0.8658)

F1_rescaled
model      prompt        correct perm_glb perm_dom      gap
Gemini     Zero-shot      0.3942  -0.0138   0.0501   0.3440
Gemini     CoT            0.1423  -0.0881  -0.0497   0.1920
Gemini     Str

In [6]:
# Cell 6 - download the two files to send back
from google.colab import files
files.download('bertscore_results.csv')
files.download('bertscore_summary.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>